# Training the NumPy Seq2Seq + Bahdanau Attention Summarizer

This notebook is just the **training** part of the project pulled out of
`train.py` so it can be run cell-by-cell (locally in Jupyter/VS Code, or in
Google Colab).

**Running in Google Colab:**
1. On your own machine, locate `numpy_seq2seq_colab.zip` (in the project's
   root folder, next to `numpy_seq2seq/` and `archive/`) - it bundles this
   notebook's code files plus the BBC News Summary dataset (~4.6MB).
2. Upload *this notebook* to Colab (File -> Upload notebook), or upload it
   to Google Drive and open with Colab.
3. Run the cell below - it detects Colab and pops up a file-upload widget;
   pick `numpy_seq2seq_colab.zip` there. It only asks once per session.
4. Runtime type doesn't matter - a GPU gives **no speedup** here since this
   is plain NumPy with no GPU calls; CPU runtime is fine (and free-tier
   friendly).

**Running locally instead:** just open this notebook from inside the
`numpy_seq2seq/` folder (VS Code / Jupyter) - the cell below detects it's
not Colab and does nothing extra.

**Note either way:** the archive dataset must be present as
`archive/BBC News Summary/...` two directories above wherever `data.py`
ends up living, and NumPy is the model's only dependency (`matplotlib` is
used below just to plot the loss curve).

In [ ]:
import os, sys, zipfile

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists("data.py"):
    from google.colab import files
    print("Upload numpy_seq2seq_colab.zip now (see the instructions above)...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(".")
    os.chdir("numpy_seq2seq")
    print("Extracted. Now running from:", os.getcwd())
elif IN_COLAB:
    print("data.py already present, skipping upload. Running from:", os.getcwd())

sys.path.insert(0, os.getcwd())

import time
import numpy as np

from data import prepare_dataset
from model import Seq2SeqAttention
from optim import Adam, clip_grads_

In [ ]:
# Hyperparameters (same defaults as train.py --help)
VOCAB_SIZE = 8000
ENC_MAX_LEN = 60
DEC_MAX_LEN = 20
EMB_DIM = 96
HIDDEN_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 60          # upper bound - early stopping (below) will likely stop sooner
LR = 1e-3
CLIP_NORM = 5.0
SEED = 0
PATIENCE = 5         # stop after this many epochs with no validation-loss improvement
CHECKPOINT_PATH = "checkpoint.npz"

## 1. Load and preprocess the dataset

First run tokenizes the raw text files and caches the result to `data_cache.npz` (fast on later runs).

In [ ]:
ds = prepare_dataset(vocab_size=VOCAB_SIZE, enc_max_len=ENC_MAX_LEN, dec_max_len=DEC_MAX_LEN)
print(f"train examples: {len(ds['enc_ids_train'])}, val examples: {len(ds['enc_ids_val'])}")
print(f"vocab size: {len(ds['itos'])}")

## 2. Build the model and optimizer

In [ ]:
model = Seq2SeqAttention(vocab_size=len(ds["itos"]), emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE, seed=SEED)
optimizer = Adam(model.params, lr=LR)
rng = np.random.RandomState(SEED)

In [ ]:
def iterate_batches(enc_ids, dec_ids, batch_size, rng, shuffle=True):
    n = enc_ids.shape[0]
    order = rng.permutation(n) if shuffle else np.arange(n)
    for start in range(0, n, batch_size):
        idx = order[start:start + batch_size]
        yield enc_ids[idx], dec_ids[idx]

## 3. Training loop

Each batch: forward pass (encoder -> attention -> decoder) -> manual backward pass (BPTT) -> gradient clipping -> Adam step. Saves a checkpoint after every epoch.

In [ ]:
def evaluate(model, enc_ids, dec_ids, batch_size, rng):
    """Validation-set loss (no backward pass) - the honest signal for whether
    the model is generalizing, since training loss keeps dropping even after
    the model starts just memorizing the training set."""
    total_loss_tokens = 0.0
    total_tokens = 0.0
    for enc_batch, dec_batch in iterate_batches(enc_ids, dec_ids, batch_size, rng, shuffle=False):
        avg_loss, num_real, _ = model.forward(enc_batch, dec_batch)
        total_loss_tokens += avg_loss * num_real
        total_tokens += num_real
    return total_loss_tokens / max(total_tokens, 1.0)


enc_ids_train, dec_ids_train = ds["enc_ids_train"], ds["dec_ids_train"]

train_loss_history = []
val_loss_history = []
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    total_loss_tokens = 0.0
    total_tokens = 0.0
    for enc_batch, dec_batch in iterate_batches(enc_ids_train, dec_ids_train, BATCH_SIZE, rng):
        avg_loss, num_real, cache = model.forward(enc_batch, dec_batch)
        grads = model.backward(cache)
        clip_grads_(grads, max_norm=CLIP_NORM)
        optimizer.step(model.params, grads)

        total_loss_tokens += avg_loss * num_real
        total_tokens += num_real

    train_loss = total_loss_tokens / total_tokens
    val_loss = evaluate(model, ds["enc_ids_val"], ds["dec_ids_val"], BATCH_SIZE, rng)
    train_loss_history.append(train_loss)
    val_loss_history.append(val_loss)

    elapsed = time.time() - epoch_start
    print(f"epoch {epoch}/{EPOCHS} - train_loss={train_loss:.4f} val_loss={val_loss:.4f} ({elapsed:.1f}s)")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        model.save(CHECKPOINT_PATH)
        print(f"  -> new best val_loss, saved checkpoint")
    else:
        epochs_without_improvement += 1
        print(f"  -> no val_loss improvement ({epochs_without_improvement}/{PATIENCE})")

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: no val_loss improvement for {PATIENCE} epochs in a row.")
        break

print(f"Best val_loss={best_val_loss:.4f}, checkpoint saved to {CHECKPOINT_PATH}")

## 4. Plot the training vs. validation loss

If validation loss flattens out or rises while training loss keeps falling, that gap *is* overfitting - the model memorizing the training set instead of generalizing.

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(train_loss_history) + 1)
plt.figure(figsize=(6, 4))
plt.plot(epochs_ran, train_loss_history, marker="o", label="train loss")
plt.plot(epochs_ran, val_loss_history, marker="o", label="val loss")
plt.xlabel("epoch")
plt.ylabel("avg loss (cross-entropy per token)")
plt.title("Training vs. validation loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Quick sanity check: summarize a few validation articles

In [ ]:
def ids_to_text(ids, itos):
    words = []
    for i in ids:
        tok = itos[i]
        if tok == "<eos>":
            break
        if tok in ("<pad>", "<sos>"):
            continue
        words.append(tok)
    return " ".join(words)

sample_enc = ds["enc_ids_val"][:3]
sample_ref = ds["dec_ids_val"][:3]
pred_ids, _ = model.greedy_decode(sample_enc, max_len=DEC_MAX_LEN)
for i in range(3):
    print("reference :", ids_to_text(sample_ref[i], ds["itos"]))
    print("prediction:", ids_to_text(pred_ids[i], ds["itos"]))
    print()